# Phase 2: Computation (Assembled)
Runs all 6 builders via ComputationAssembler.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd() if Path.cwd().name == 'certification_framework' else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
import json
from scripts.computation.assembler import ComputationAssembler

NB_DATA = ROOT / 'notebooks' / 'data'
phase1_path = NB_DATA / 'parsed_context.json'

assembler = ComputationAssembler(phase1_path)
result = assembler.assemble()

# Save to notebook-local data
(NB_DATA / 'computed_content.json').write_text(
    json.dumps(result, indent=2, default=str, ensure_ascii=False), encoding='utf-8'
)

print(f'Top-level keys: {list(result.keys())}')
print(f'\nScorecard: {len(result["scorecard"]["dimensions"])} dimensions')
for d in result['scorecard']['dimensions']:
    print(f'  {d["dimension"]}: {d["value"]}')
print(f'\nTables: {len(result["tables"])}')
for name in result['tables']:
    print(f'  {name}: {len(result["tables"][name].get("rows", []))} rows')
print(f'\nCharts: {len(result["charts"])}')
for name in result['charts']:
    print(f'  {name}: {result["charts"][name]["chart_type"]}')
print(f'\nFindings: {len(result["findings"])}')
print(f'Assessments: {len(result["assessments"])} categories')
print(f'Cards: {len(result["cards"])}')

In [ ]:
raw = json.loads((ROOT / 'data' / 'input' / 'aggregated_scorecard_output.json').read_text(encoding='utf-8'))
phase1 = json.loads(phase1_path.read_text(encoding='utf-8'))

print('=== Validation: Computed vs Input ===')
all_ok = True

# Check category count matches assessments
cat_count = len(raw['fault_category_scorecards'])
assess_count = len(result['assessments'])
ok = 'PASS' if assess_count == cat_count else 'FAIL'
if ok == 'FAIL': all_ok = False
print(f'  {ok} assessment categories: {assess_count} (expected {cat_count})')

# Check scorecard dimension count
dim_count = len(result['scorecard']['dimensions'])
ok = 'PASS' if dim_count >= 7 else 'FAIL'
if ok == 'FAIL': all_ok = False
print(f'  {ok} scorecard dimensions: {dim_count} (expected >= 7)')

# Validate scorecard values are in 0-1 range
for d in result['scorecard']['dimensions']:
    v = d['value']
    ok = 'PASS' if 0 <= v <= 1 else 'FAIL'
    if ok == 'FAIL': all_ok = False
    print(f'  {ok} {d["dimension"]}: {v} (in [0,1])')

# Validate table row counts against categories
for tbl_name in ['ttd_stats', 'ttm_stats', 'detection_rates']:
    if tbl_name in result['tables']:
        rows = len(result['tables'][tbl_name].get('rows', []))
        ok = 'PASS' if rows == cat_count else 'FAIL'
        if ok == 'FAIL': all_ok = False
        print(f'  {ok} {tbl_name} rows: {rows} (expected {cat_count})')

print(f'\nResult: {"ALL CHECKS PASSED" if all_ok else "SOME CHECKS FAILED"}')